# Sentinel-2 4 km Spatial-Aggregate SIF Model

Train a three-downsampling U-Net using 4 km x 4 km predictor chips at 20 m resolution. Each sample has one aggregate SIF target and one equal-footprint supervision weight map. The model predicts a 20 m SIF map; its weighted mean over the sampled footprint support is compared with the spatially aggregated OCO-2 SIF target.

## 1. Imports

In [ ]:
from pathlib import Path
from collections import OrderedDict
import hashlib
import json
import math
import random

import altair as alt
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset, Sampler

alt.data_transformers.disable_max_rows()
pd.set_option('display.max_columns', 100)

## 2. Configuration

In [ ]:
# Edit CHIP_DIR after attaching the Kaggle dataset.
CHIP_DIR = Path(
    '/kaggle/input/CHANGE_ME/spatial_aggregate_4km_20m_indices_fapar_par_apar_active_crop'
)
METADATA_PATH = CHIP_DIR / 'chip_metadata.csv'
FOOTPRINT_METADATA_PATH = CHIP_DIR / 'footprint_metadata.csv'

OUTPUT_DIR = Path('/kaggle/working/sentinel2_spatial_aggregate_4km_model')
STATS_CACHE_DIR = Path('/kaggle/working/normalization_stats')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
STATS_CACHE_DIR.mkdir(parents=True, exist_ok=True)

TARGET_COL = 'aggregated_target_modis_sif'

SEED = 42
TRAIN_FRAC = 0.80
VAL_FRAC = 0.10
TEST_FRAC = 0.10

BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = 4
BASE_CHANNELS = 16
EPOCHS = 30
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
HUBER_BETA = 1.0

EARLY_STOPPING_PATIENCE = 7
LR_PATIENCE = 3
LR_FACTOR = 0.5
MIN_LEARNING_RATE = 1e-6

NUM_WORKERS = 2
SHARD_CACHE_SIZE = 8
MAX_SAMPLES_FOR_STATS = 256
USE_STATS_CACHE = True
NORMALIZE_TARGET = True
USE_AMP = True

MIN_FOOTPRINTS = 5
SIF_BIN_EDGES = [-0.5, -0.25, 0.0, 0.25, 0.5, 0.75, 1.0, 1.25]
SIF_BIN_LABELS = [
    '[-0.5,-0.25)', '[-0.25,0)', '[0,0.25)', '[0.25,0.5)',
    '[0.5,0.75)', '[0.75,1)', '[1,1.25]'
]

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
AMP_ENABLED = bool(USE_AMP and DEVICE.type == 'cuda')

print('device:', DEVICE)
print('AMP enabled:', AMP_ENABLED)
print('effective batch size:', BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS)

## 3. Index Shards and Load Metadata

In [ ]:
def decode_strings(values: np.ndarray) -> list[str]:
    output = []
    for value in values:
        if isinstance(value, bytes):
            output.append(value.decode('utf-8'))
        else:
            output.append(str(value))
    return output


def build_shard_index(chip_dir: Path) -> tuple[pd.DataFrame, list[str], list[str]]:
    shard_files = sorted(chip_dir.glob('chips_*.npz'))
    if not shard_files:
        raise FileNotFoundError(f'No chips_*.npz files found in {chip_dir}')

    rows = []
    channel_names = None
    target_names = None
    sample_order = 0

    for shard_id, shard_path in enumerate(shard_files):
        with np.load(shard_path, allow_pickle=False) as shard:
            aggregation_ids = decode_strings(shard['aggregation_id'])
            current_channels = decode_strings(shard['channel_names'])
            current_targets = decode_strings(shard['target_name'])

        if channel_names is None:
            channel_names = current_channels
            target_names = current_targets
        elif current_channels != channel_names or current_targets != target_names:
            raise ValueError(f'Inconsistent shard schema in {shard_path}')

        for local_index, aggregation_id in enumerate(aggregation_ids):
            rows.append({
                'sample_order': sample_order,
                'shard_id': shard_id,
                'shard_path': str(shard_path),
                'local_index': local_index,
                'aggregation_id': aggregation_id,
            })
            sample_order += 1

    shard_index = pd.DataFrame(rows)
    if shard_index['aggregation_id'].duplicated().any():
        raise ValueError(
            'Duplicate aggregation IDs found. Remove stale NPZ files and rebuild the dataset.'
        )
    return shard_index, channel_names, target_names


metadata = pd.read_csv(METADATA_PATH, parse_dates=['Delta_Date', 'par_date'])
footprint_metadata = pd.read_csv(FOOTPRINT_METADATA_PATH, parse_dates=['Delta_Date'])
shard_index, channel_names, target_names = build_shard_index(CHIP_DIR)

samples = metadata.merge(
    shard_index,
    on='aggregation_id',
    how='inner',
    validate='one_to_one',
)

if target_names != [TARGET_COL]:
    raise ValueError(f'Expected shard target {TARGET_COL}, found {target_names}')
if len(channel_names) != 19:
    raise ValueError(f'Expected 19 predictor channels, found {len(channel_names)}')

print('metadata rows:', len(metadata))
print('indexed shard rows:', len(shard_index))
print('merged samples:', len(samples))
print('channels:', len(channel_names), channel_names)
print('targets:', target_names)
samples.head()

## 4. Filter and Inspect Aggregate Samples

In [ ]:
samples['y_aggregate'] = pd.to_numeric(samples['y_aggregate'], errors='coerce')
samples['n_footprints'] = pd.to_numeric(samples['n_footprints'], errors='coerce')
samples['measurement_mode'] = pd.to_numeric(
    samples['measurement_mode'], errors='coerce'
)
samples['month'] = samples['Delta_Date'].dt.month.astype(int)

samples = samples[
    (samples['n_footprints'] >= MIN_FOOTPRINTS)
    & np.isfinite(samples['y_aggregate'])
    & samples['product_path'].notna()
].copy()

samples = samples.sort_values(
    ['sif_year', 'sif_doy', 'mgrs_tile_t', 'aggregation_id']
).reset_index(drop=True)

required_validity = [
    'ndmi_valid_fraction', 'ndvi_valid_fraction', 'evi_valid_fraction',
    'nirv_valid_fraction', 'ndre_valid_fraction', 'fapar_valid_fraction',
    'par_valid_fraction', 'apar_valid_fraction'
]
missing_validity = [column for column in required_validity if column not in samples.columns]
if missing_validity:
    raise ValueError(f'Missing predictor-validity columns: {missing_validity}')

print(f'Kept {len(samples):,} aggregate chips')
display(
    samples[
        ['n_footprints', *required_validity, 'y_aggregate']
    ].describe().T
)
display(
    samples.groupby(
        ['sif_year', 'month', 'measurement_mode'], dropna=False
    ).size().rename('n_chips').reset_index()
)

## 5. Component-Preserving, Chip-Count-Balanced Train/Validation/Test Split

Rows connected by either the same Delta_Date or the same Sentinel product_path form one indivisible component. Complete components are assigned to approximately 80/10/10 of chips while balancing month, measurement mode, and MGRS tile. Year is not used for balancing.

In [ ]:
def add_leakage_components(table: pd.DataFrame) -> pd.DataFrame:
    table = table.reset_index(drop=True).copy()
    n_rows = len(table)
    parent = np.arange(n_rows, dtype=np.int64)
    rank = np.zeros(n_rows, dtype=np.int8)

    def find(index: int) -> int:
        while parent[index] != index:
            parent[index] = parent[parent[index]]
            index = int(parent[index])
        return index

    def union(left: int, right: int) -> None:
        root_left = find(left)
        root_right = find(right)
        if root_left == root_right:
            return
        if rank[root_left] < rank[root_right]:
            root_left, root_right = root_right, root_left
        parent[root_right] = root_left
        if rank[root_left] == rank[root_right]:
            rank[root_left] += 1

    for column in ['Delta_Date', 'product_path']:
        for indices in table.groupby(column, dropna=False).indices.values():
            indices = np.asarray(indices, dtype=np.int64)
            first = int(indices[0])
            for index in indices[1:]:
                union(first, int(index))

    roots = np.asarray([find(index) for index in range(n_rows)])
    unique_roots = sorted(np.unique(roots).tolist())
    root_names = {
        root: f'component_{number:05d}'
        for number, root in enumerate(unique_roots)
    }
    table['split_component'] = [root_names[int(root)] for root in roots]
    return table


def component_summary_table(table: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for component, group in table.groupby('split_component', sort=False):
        months = sorted(group['month'].dropna().astype(int).unique().tolist())
        modes = sorted(
            group['measurement_mode'].dropna().astype(int).unique().tolist()
        )
        tiles = sorted(group['mgrs_tile_t'].dropna().astype(str).unique().tolist())
        rows.append({
            'split_component': component,
            'n_chips': len(group),
            'n_dates': group['Delta_Date'].nunique(),
            'n_products': group['product_path'].nunique(),
            'months': ';'.join(map(str, months)),
            'measurement_modes': ';'.join(map(str, modes)),
            'mgrs_tiles': ';'.join(tiles),
            'n_months': len(months),
            'n_modes': len(modes),
            'n_mgrs_tiles': len(tiles),
        })
    return pd.DataFrame(rows)


def build_component_balance_vectors(
    table: pd.DataFrame,
    components: pd.DataFrame,
    balance_columns: list[str],
) -> tuple[np.ndarray, list[str], np.ndarray, dict[str, list]]:
    component_ids = components['split_component'].tolist()
    grouped = {
        component: group
        for component, group in table.groupby('split_component', sort=False)
    }
    balance_levels = {
        column: sorted(
            table[column].dropna().unique().tolist(),
            key=lambda value: str(value),
        )
        for column in balance_columns
    }

    feature_labels = ['all_chips']
    feature_weights = [4.0]
    for column in balance_columns:
        levels = balance_levels[column]
        feature_labels.extend(
            [f'{column}={level}' for level in levels]
        )
        # Give month, mode, and tile equal total influence regardless of
        # how many levels each variable contains.
        feature_weights.extend([1.0 / len(levels)] * len(levels))

    vectors = np.zeros(
        (len(component_ids), len(feature_labels)),
        dtype=np.float64,
    )
    for component_index, component in enumerate(component_ids):
        group = grouped[component]
        vectors[component_index, 0] = len(group)
        feature_index = 1
        for column in balance_columns:
            counts = group[column].value_counts(dropna=False)
            for level in balance_levels[column]:
                vectors[component_index, feature_index] = float(
                    counts.get(level, 0)
                )
                feature_index += 1

    return (
        vectors,
        feature_labels,
        np.asarray(feature_weights, dtype=np.float64),
        balance_levels,
    )


def grouped_balanced_component_split(
    table: pd.DataFrame,
    n_restarts: int = 2000,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    table = add_leakage_components(table)
    components = component_summary_table(table)
    balance_columns = ['month', 'measurement_mode', 'mgrs_tile_t']
    (
        component_vectors,
        feature_labels,
        feature_weights,
        balance_levels,
    ) = build_component_balance_vectors(
        table,
        components,
        balance_columns,
    )

    # Every required level must occur in at least three separate components
    # before it can appear in all three splits.
    for column, levels in balance_levels.items():
        presence = (
            table[['split_component', column]]
            .drop_duplicates()
            .groupby(column, dropna=False)
            .size()
        )
        impossible = {
            str(level): int(presence.get(level, 0))
            for level in levels
            if int(presence.get(level, 0)) < 3
        }
        if impossible:
            raise ValueError(
                f'Cannot represent every {column} level in all splits. '
                f'Component counts: {impossible}'
            )

    split_names = np.asarray(['train', 'validation', 'test'])
    split_fractions = np.asarray(
        [TRAIN_FRAC, VAL_FRAC, TEST_FRAC],
        dtype=np.float64,
    )
    if np.any(split_fractions <= 0) or not np.isclose(
        split_fractions.sum(), 1.0
    ):
        raise ValueError(f'Invalid split fractions: {split_fractions}')

    feature_totals = component_vectors.sum(axis=0)
    target_counts = split_fractions[:, None] * feature_totals[None, :]
    target_scale = np.maximum(target_counts, 1.0)

    def balance_error(split_counts: np.ndarray) -> float:
        relative_error = (split_counts - target_counts) / target_scale
        return float(
            np.sum(
                feature_weights[None, :] * relative_error ** 2
            )
        )

    def final_objective(split_counts: np.ndarray) -> float:
        missing_required_levels = int(
            np.count_nonzero(split_counts[:, 1:] <= 0)
        )
        return (
            balance_error(split_counts)
            + 1_000_000.0 * missing_required_levels
        )

    rng = np.random.default_rng(SEED)
    best_assignments = None
    best_counts = None
    best_score = np.inf

    for _ in range(n_restarts):
        # Usually place large components first, with enough jitter for the
        # repeated searches to explore different valid allocations.
        priority = (
            component_vectors[:, 0]
            * rng.uniform(0.75, 1.25, size=len(components))
        )
        component_order = np.argsort(-priority)
        assignments = np.full(len(components), -1, dtype=np.int8)
        split_counts = np.zeros_like(target_counts)

        for component_index in component_order:
            candidate_scores = np.empty(len(split_names), dtype=np.float64)
            for split_index in range(len(split_names)):
                candidate_counts = split_counts.copy()
                candidate_counts[split_index] += component_vectors[
                    component_index
                ]
                candidate_scores[split_index] = balance_error(
                    candidate_counts
                )

            minimum = candidate_scores.min()
            best_candidates = np.flatnonzero(
                np.isclose(candidate_scores, minimum, rtol=1e-12, atol=1e-12)
            )
            chosen_split = int(rng.choice(best_candidates))
            assignments[component_index] = chosen_split
            split_counts[chosen_split] += component_vectors[component_index]

        score = final_objective(split_counts)
        if score < best_score:
            best_score = score
            best_assignments = assignments.copy()
            best_counts = split_counts.copy()

    if best_assignments is None or best_counts is None:
        raise RuntimeError('No component allocation was produced.')
    if np.any(best_counts[:, 1:] <= 0):
        raise RuntimeError(
            'Best allocation is missing at least one required month, mode, '
            'or MGRS tile.'
        )

    component_to_split = {
        component: split_names[int(split_index)]
        for component, split_index in zip(
            components['split_component'],
            best_assignments,
        )
    }
    table['split'] = table['split_component'].map(component_to_split)
    if table['split'].isna().any():
        raise RuntimeError('At least one leakage component was not assigned.')

    train = table[table['split'] == 'train'].copy().reset_index(drop=True)
    validation = (
        table[table['split'] == 'validation']
        .copy()
        .reset_index(drop=True)
    )
    test = table[table['split'] == 'test'].copy().reset_index(drop=True)
    if train.empty or validation.empty or test.empty:
        raise ValueError('Grouped split produced an empty split.')

    print('Best component-balance objective:', best_score)
    target_and_actual = pd.DataFrame({
        'split': split_names,
        'target_chips': target_counts[:, 0],
        'actual_chips': best_counts[:, 0].astype(int),
    })
    target_and_actual['actual_fraction'] = (
        target_and_actual['actual_chips'] / len(table)
    )
    display(target_and_actual)

    return train, validation, test, components


def assert_disjoint_values(
    left: pd.DataFrame,
    right: pd.DataFrame,
    column: str,
    label: str,
) -> None:
    shared = set(left[column].astype(str)) & set(right[column].astype(str))
    if shared:
        raise RuntimeError(f'{label} leakage detected: {len(shared)} shared values')


def assert_all_levels_present(
    full_table: pd.DataFrame,
    split_tables: dict[str, pd.DataFrame],
    column: str,
) -> None:
    expected = set(full_table[column].dropna().astype(str))
    for split_name, split_table in split_tables.items():
        observed = set(split_table[column].dropna().astype(str))
        missing = sorted(expected - observed)
        if missing:
            raise RuntimeError(
                f'{split_name} is missing {column} levels: {missing}'
            )


def marginal_split_summary(
    table: pd.DataFrame,
    column: str,
) -> pd.DataFrame:
    summary = (
        table.groupby(['split', column], dropna=False)
        .size()
        .rename('n_chips')
        .reset_index()
    )
    level_totals = (
        table.groupby(column, dropna=False)
        .size()
        .rename('level_total')
        .reset_index()
    )
    summary = summary.merge(level_totals, on=column, how='left')
    summary['fraction_of_level'] = (
        summary['n_chips'] / summary['level_total']
    )
    return summary.sort_values([column, 'split']).reset_index(drop=True)


train_table, val_table, test_table, split_components = (
    grouped_balanced_component_split(samples)
)

split_tables = {
    'train': train_table,
    'validation': val_table,
    'test': test_table,
}
for column in ['month', 'measurement_mode', 'mgrs_tile_t']:
    assert_all_levels_present(samples, split_tables, column)

for left_name, left, right_name, right in [
    ('train', train_table, 'validation', val_table),
    ('train', train_table, 'test', test_table),
    ('validation', val_table, 'test', test_table),
]:
    assert_disjoint_values(
        left, right, 'Delta_Date', f'{left_name}/{right_name} date'
    )
    assert_disjoint_values(
        left,
        right,
        'product_path',
        f'{left_name}/{right_name} Sentinel product',
    )

print(
    'train chips:', len(train_table),
    'dates:', train_table['Delta_Date'].nunique(),
    'products:', train_table['product_path'].nunique(),
)
print(
    'validation chips:', len(val_table),
    'dates:', val_table['Delta_Date'].nunique(),
    'products:', val_table['product_path'].nunique(),
)
print(
    'test chips:', len(test_table),
    'dates:', test_table['Delta_Date'].nunique(),
    'products:', test_table['product_path'].nunique(),
)

all_split_rows = pd.concat(
    [train_table, val_table, test_table],
    ignore_index=True,
)
for column in ['month', 'measurement_mode', 'mgrs_tile_t']:
    print(f'\nBalance by {column}:')
    display(marginal_split_summary(all_split_rows, column))

display(split_components.describe(include='all').T)

In [ ]:
component_split_lookup = pd.concat([
    train_table[['split_component']].assign(split='train'),
    val_table[['split_component']].assign(split='validation'),
    test_table[['split_component']].assign(split='test'),
]).drop_duplicates('split_component')
split_components = split_components.merge(
    component_split_lookup,
    on='split_component',
    how='left',
    validate='one_to_one',
)
if split_components['split'].isna().any():
    raise RuntimeError('A split component is missing its split label.')
display(split_components.groupby('split')['n_chips'].agg(['count', 'sum']))

## 6. Dataset, Shard Cache, and NaN-Aware Normalization

In [ ]:
class NpzShardCache:
    def __init__(self, max_size: int):
        self.max_size = max(1, int(max_size))
        self.cache: OrderedDict[str, dict[str, np.ndarray]] = OrderedDict()

    def get(self, path: str) -> dict[str, np.ndarray]:
        if path in self.cache:
            self.cache.move_to_end(path)
            return self.cache[path]

        with np.load(path, allow_pickle=False) as shard:
            loaded = {
                'X': shard['X'].copy(),
                'aggregate_weight_map': shard['aggregate_weight_map'].copy(),
                'y_aggregate': shard['y_aggregate'].copy(),
                'n_footprints': shard['n_footprints'].copy(),
            }

        self.cache[path] = loaded
        self.cache.move_to_end(path)
        while len(self.cache) > self.max_size:
            self.cache.popitem(last=False)
        return loaded


class SentinelAggregateDataset(Dataset):
    def __init__(
        self,
        table: pd.DataFrame,
        channel_mean: np.ndarray | None = None,
        channel_std: np.ndarray | None = None,
        target_mean: float | None = None,
        target_std: float | None = None,
        cache_size: int = 8,
    ):
        self.table = table.reset_index(drop=True).copy()
        self.channel_mean = channel_mean
        self.channel_std = channel_std
        self.target_mean = target_mean
        self.target_std = target_std
        self.cache = NpzShardCache(cache_size)

    def __len__(self) -> int:
        return len(self.table)

    def __getitem__(self, index: int):
        row = self.table.iloc[index]
        shard = self.cache.get(str(row['shard_path']))
        local_index = int(row['local_index'])

        # Cast stored float16 arrays to float32 before normalization and training.
        x = shard['X'][local_index].astype(np.float32, copy=True)
        weight = shard['aggregate_weight_map'][local_index].astype(
            np.float32, copy=True
        )
        y = np.float32(shard['y_aggregate'][local_index])
        n_footprints = int(shard['n_footprints'][local_index])

        weight = np.nan_to_num(weight, nan=0.0, posinf=0.0, neginf=0.0)
        weight_sum = float(weight.sum(dtype=np.float64))
        if not np.isfinite(weight_sum) or weight_sum <= 0:
            raise ValueError(f'Invalid aggregate weight map at dataset index {index}')
        weight /= weight_sum

        if self.channel_mean is not None and self.channel_std is not None:
            x = (
                x - self.channel_mean[:, None, None]
            ) / self.channel_std[:, None, None]
            x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)

        if self.target_mean is not None and self.target_std is not None:
            y = np.float32((y - self.target_mean) / self.target_std)

        return (
            torch.from_numpy(x),
            torch.from_numpy(weight),
            torch.tensor(y, dtype=torch.float32),
            torch.tensor(n_footprints, dtype=torch.int16),
            torch.tensor(index, dtype=torch.long),
        )


def make_stats_cache_path(table: pd.DataFrame) -> Path:
    fields = [
        f'target={TARGET_COL}',
        f'seed={SEED}',
        f'max_stats={MAX_SAMPLES_FOR_STATS}',
        'channels=' + ','.join(channel_names),
        *sorted(table['aggregation_id'].astype(str).tolist()),
    ]
    digest = hashlib.md5('\n'.join(fields).encode('utf-8')).hexdigest()[:16]
    return STATS_CACHE_DIR / f'sentinel2_4km_normalization_{digest}.npz'


def compute_channel_stats(
    table: pd.DataFrame,
    max_samples: int,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    rng = np.random.default_rng(SEED)
    if len(table) > max_samples:
        selected = rng.choice(len(table), size=max_samples, replace=False)
        stat_table = table.iloc[selected].copy().reset_index(drop=True)
    else:
        stat_table = table.copy().reset_index(drop=True)

    dataset = SentinelAggregateDataset(stat_table, cache_size=SHARD_CACHE_SIZE)
    n_channels = len(channel_names)
    channel_sum = np.zeros(n_channels, dtype=np.float64)
    channel_sumsq = np.zeros(n_channels, dtype=np.float64)
    channel_count = np.zeros(n_channels, dtype=np.int64)

    for index in range(len(dataset)):
        x, _, _, _, _ = dataset[index]
        array = x.numpy()
        finite = np.isfinite(array)
        safe = np.where(finite, array, 0.0)
        channel_sum += safe.sum(axis=(1, 2), dtype=np.float64)
        channel_sumsq += (safe * safe).sum(axis=(1, 2), dtype=np.float64)
        channel_count += finite.sum(axis=(1, 2))

        if (index + 1) % 32 == 0 or index + 1 == len(dataset):
            print(f'Normalization stats: {index + 1} / {len(dataset)} chips')

    if (channel_count == 0).any():
        bad = np.asarray(channel_names)[channel_count == 0].tolist()
        raise ValueError(f'Channels have no valid training pixels: {bad}')

    mean = channel_sum / channel_count
    variance = np.maximum(channel_sumsq / channel_count - mean ** 2, 1e-8)
    std = np.sqrt(variance)
    return mean.astype(np.float32), std.astype(np.float32), channel_count


stats_cache_path = make_stats_cache_path(train_table)

if USE_STATS_CACHE and stats_cache_path.exists():
    print('Loading normalization stats:', stats_cache_path)
    with np.load(stats_cache_path, allow_pickle=False) as stats:
        cached_channels = decode_strings(stats['channel_names'])
        if cached_channels != channel_names:
            raise ValueError('Cached channel names do not match the dataset.')
        channel_mean = stats['channel_mean'].astype(np.float32)
        channel_std = stats['channel_std'].astype(np.float32)
        channel_count = stats['channel_count'].astype(np.int64)
        target_mean = float(stats['target_mean'])
        target_std = float(stats['target_std'])
else:
    print('Computing NaN-aware statistics from training chips...')
    channel_mean, channel_std, channel_count = compute_channel_stats(
        train_table,
        MAX_SAMPLES_FOR_STATS,
    )
    training_targets = train_table['y_aggregate'].to_numpy(dtype=np.float64)
    training_targets = training_targets[np.isfinite(training_targets)]
    target_mean = float(training_targets.mean())
    target_std = float(training_targets.std(ddof=1))
    if not np.isfinite(target_std) or target_std <= 0:
        raise ValueError(f'Invalid target standard deviation: {target_std}')

    np.savez_compressed(
        stats_cache_path,
        channel_names=np.asarray(channel_names),
        channel_mean=channel_mean,
        channel_std=channel_std,
        channel_count=channel_count,
        target_mean=np.asarray(target_mean, dtype=np.float32),
        target_std=np.asarray(target_std, dtype=np.float32),
    )
    print('Saved normalization stats:', stats_cache_path)

normalization_table = pd.DataFrame({
    'channel': channel_names,
    'mean': channel_mean,
    'std': channel_std,
    'valid_pixel_count': channel_count,
})
display(normalization_table)
print('target mean:', target_mean)
print('target std:', target_std)

## 7. Shard-Aware DataLoaders

In [ ]:
class ShardBatchSampler(Sampler[list[int]]):
    def __init__(
        self,
        table: pd.DataFrame,
        batch_size: int,
        shuffle: bool,
        seed: int,
    ):
        self.table = table.reset_index(drop=True)
        self.batch_size = int(batch_size)
        self.shuffle = bool(shuffle)
        self.seed = int(seed)
        self.epoch = 0
        self.groups = [
            np.asarray(indices, dtype=np.int64)
            for indices in self.table.groupby(
                'shard_path', sort=False
            ).indices.values()
        ]

    def set_epoch(self, epoch: int) -> None:
        self.epoch = int(epoch)

    def __iter__(self):
        rng = np.random.default_rng(self.seed + self.epoch)
        group_order = np.arange(len(self.groups))
        if self.shuffle:
            rng.shuffle(group_order)

        for group_index in group_order:
            indices = self.groups[group_index].copy()
            if self.shuffle:
                rng.shuffle(indices)
            for start in range(0, len(indices), self.batch_size):
                yield indices[start:start + self.batch_size].tolist()

    def __len__(self) -> int:
        return sum(
            math.ceil(len(indices) / self.batch_size)
            for indices in self.groups
        )


def seed_worker(worker_id: int) -> None:
    worker_seed = SEED + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)


dataset_kwargs = dict(
    channel_mean=channel_mean,
    channel_std=channel_std,
    target_mean=target_mean if NORMALIZE_TARGET else None,
    target_std=target_std if NORMALIZE_TARGET else None,
    cache_size=SHARD_CACHE_SIZE,
)

train_dataset = SentinelAggregateDataset(train_table, **dataset_kwargs)
val_dataset = SentinelAggregateDataset(val_table, **dataset_kwargs)
test_dataset = SentinelAggregateDataset(test_table, **dataset_kwargs)

train_batch_sampler = ShardBatchSampler(
    train_table, BATCH_SIZE, shuffle=True, seed=SEED
)
val_batch_sampler = ShardBatchSampler(
    val_table, BATCH_SIZE, shuffle=False, seed=SEED
)
test_batch_sampler = ShardBatchSampler(
    test_table, BATCH_SIZE, shuffle=False, seed=SEED
)

loader_kwargs = {
    'num_workers': NUM_WORKERS,
    'pin_memory': DEVICE.type == 'cuda',
    'worker_init_fn': seed_worker,
}
if NUM_WORKERS > 0:
    loader_kwargs.update({'persistent_workers': True, 'prefetch_factor': 2})

train_loader = DataLoader(
    train_dataset, batch_sampler=train_batch_sampler, **loader_kwargs
)
val_loader = DataLoader(
    val_dataset, batch_sampler=val_batch_sampler, **loader_kwargs
)
test_loader = DataLoader(
    test_dataset, batch_sampler=test_batch_sampler, **loader_kwargs
)

print('train batches:', len(train_loader))
print('validation batches:', len(val_loader))
print('test batches:', len(test_loader))

## 8. Three-Downsampling U-Net

In [ ]:
def group_count(channels: int) -> int:
    for groups in (8, 4, 2, 1):
        if channels % groups == 0:
            return groups
    return 1


class ConvBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        groups = group_count(out_channels)
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            nn.GroupNorm(groups, out_channels),
            nn.SiLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.GroupNorm(groups, out_channels),
            nn.SiLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class ThreeLevelUNet(nn.Module):
    def __init__(self, in_channels: int, base_channels: int = 16):
        super().__init__()
        self.enc1 = ConvBlock(in_channels, base_channels)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = ConvBlock(base_channels, base_channels * 2)
        self.pool2 = nn.MaxPool2d(2)
        self.enc3 = ConvBlock(base_channels * 2, base_channels * 4)
        self.pool3 = nn.MaxPool2d(2)

        self.bottleneck = ConvBlock(base_channels * 4, base_channels * 8)

        self.up3 = nn.ConvTranspose2d(
            base_channels * 8, base_channels * 4, 2, stride=2
        )
        self.dec3 = ConvBlock(base_channels * 8, base_channels * 4)
        self.up2 = nn.ConvTranspose2d(
            base_channels * 4, base_channels * 2, 2, stride=2
        )
        self.dec2 = ConvBlock(base_channels * 4, base_channels * 2)
        self.up1 = nn.ConvTranspose2d(
            base_channels * 2, base_channels, 2, stride=2
        )
        self.dec1 = ConvBlock(base_channels * 2, base_channels)
        self.out = nn.Conv2d(base_channels, 1, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        e3 = self.enc3(self.pool2(e2))
        bottleneck = self.bottleneck(self.pool3(e3))

        d3 = self.dec3(torch.cat([self.up3(bottleneck), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.out(d1)


model = ThreeLevelUNet(
    in_channels=len(channel_names),
    base_channels=BASE_CHANNELS,
).to(DEVICE)

n_parameters = sum(parameter.numel() for parameter in model.parameters())
print(model)
print(f'trainable parameters: {n_parameters:,}')

## 9. Aggregate Prediction, Loss, Calibration, and Metrics

In [ ]:
def aggregate_prediction(
    pred_map: torch.Tensor,
    weight_map: torch.Tensor,
) -> torch.Tensor:
    # pred_map:   [batch, 1, height, width]
    # weight_map: [batch, height, width], normalized to sum to one
    return (pred_map[:, 0].float() * weight_map.float()).sum(dim=(-2, -1))


def denormalize_sif(values: np.ndarray) -> np.ndarray:
    values = np.asarray(values)
    if NORMALIZE_TARGET:
        return values * target_std + target_mean
    return values


def regression_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    valid = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true = y_true[valid]
    y_pred = y_pred[valid]

    if y_true.size == 0:
        return {'n': 0, 'rmse': np.nan, 'mae': np.nan, 'bias': np.nan, 'r2': np.nan}

    residual = y_pred - y_true
    ss_res = np.sum(residual ** 2)
    ss_tot = np.sum((y_true - y_true.mean()) ** 2)
    r2 = np.nan if ss_tot <= 0 else 1.0 - ss_res / ss_tot
    return {
        'n': int(y_true.size),
        'rmse': float(np.sqrt(np.mean(residual ** 2))),
        'mae': float(np.mean(np.abs(residual))),
        'bias': float(np.mean(residual)),
        'r2': float(r2),
    }


@torch.no_grad()
def predict_table(
    model: nn.Module,
    loader: DataLoader,
    dataset: SentinelAggregateDataset,
) -> pd.DataFrame:
    model.eval()
    rows = []

    for x, weight, y, n_footprints, sample_indices in loader:
        x = x.to(DEVICE, non_blocking=True)
        weight_gpu = weight.to(DEVICE, non_blocking=True)

        with torch.autocast(
            device_type=DEVICE.type,
            dtype=torch.float16,
            enabled=AMP_ENABLED,
        ):
            pred_map = model(x)

        pred_normalized = aggregate_prediction(pred_map, weight_gpu).cpu().numpy()
        observed_normalized = y.numpy()
        predicted_sif = denormalize_sif(pred_normalized)
        observed_sif = denormalize_sif(observed_normalized)

        for batch_index, sample_index in enumerate(sample_indices.numpy()):
            sample = dataset.table.iloc[int(sample_index)]
            rows.append({
                'aggregation_id': sample['aggregation_id'],
                'cell_id': sample['cell_id'],
                'Delta_Date': sample['Delta_Date'],
                'sif_year': int(sample['sif_year']),
                'sif_doy': int(sample['sif_doy']),
                'month': int(sample['month']),
                'measurement_mode': sample['measurement_mode'],
                'mgrs_tile_t': sample['mgrs_tile_t'],
                'product_path': sample['product_path'],
                'split_component': sample['split_component'],
                'n_footprints': int(n_footprints[batch_index]),
                'states': sample.get('states', ''),
                'hzs_values': sample.get('hzs_values', ''),
                'observed_sif': float(observed_sif[batch_index]),
                'predicted_sif_raw': float(predicted_sif[batch_index]),
            })
    return pd.DataFrame(rows)


def fit_linear_calibration(
    y_true: np.ndarray,
    y_pred: np.ndarray,
) -> dict[str, float]:
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    valid = np.isfinite(y_true) & np.isfinite(y_pred)
    if valid.sum() < 2 or np.nanstd(y_pred[valid]) <= 0:
        raise ValueError('Validation predictions cannot support linear calibration.')
    design = np.column_stack([np.ones(valid.sum()), y_pred[valid]])
    intercept, slope = np.linalg.lstsq(design, y_true[valid], rcond=None)[0]
    return {'intercept': float(intercept), 'slope': float(slope)}


def apply_linear_calibration(
    y_pred: np.ndarray,
    calibration: dict[str, float],
) -> np.ndarray:
    return calibration['intercept'] + calibration['slope'] * np.asarray(y_pred)

## 10. Pre-Training Non-Finite Diagnostic

In [ ]:
# Diagnose non-finite training losses before training. This cell does not update model weights.
train_batch_sampler.set_epoch(0)
was_training = model.training
model.eval()

global_channel_absmax = np.zeros(len(channel_names), dtype=np.float64)
bad_batch_found = False
scanned_batches = 0

def tensor_is_finite(tensor: torch.Tensor) -> bool:
    return bool(torch.isfinite(tensor).all().item())


with torch.no_grad():
    for batch_no, (x, weight, y, _, _) in enumerate(train_loader, start=1):
        scanned_batches = batch_no
        finite_x_for_max = torch.nan_to_num(
            x, nan=0.0, posinf=0.0, neginf=0.0
        )
        batch_absmax = finite_x_for_max.abs().amax(
            dim=(0, 2, 3)
        ).cpu().numpy()
        global_channel_absmax = np.maximum(
            global_channel_absmax, batch_absmax
        )

        input_finite = tensor_is_finite(x)
        weight_finite = tensor_is_finite(weight)
        target_finite = tensor_is_finite(y)

        x_gpu = x.to(DEVICE, non_blocking=True)
        weight_gpu = weight.to(DEVICE, non_blocking=True)
        y_gpu = y.to(DEVICE, non_blocking=True)

        with torch.autocast(
            device_type=DEVICE.type,
            dtype=torch.float16,
            enabled=AMP_ENABLED,
        ):
            pred_map_amp = model(x_gpu)
            pred_sif_amp = aggregate_prediction(pred_map_amp, weight_gpu)
            loss_amp = F.smooth_l1_loss(
                pred_sif_amp, y_gpu, reduction='mean', beta=HUBER_BETA
            )

        amp_map_finite = tensor_is_finite(pred_map_amp)
        amp_prediction_finite = tensor_is_finite(pred_sif_amp)
        amp_loss_finite = tensor_is_finite(loss_amp)

        if not all([
            input_finite, weight_finite, target_finite,
            amp_map_finite, amp_prediction_finite, amp_loss_finite,
        ]):
            bad_batch_found = True

            with torch.autocast(
                device_type=DEVICE.type,
                enabled=False,
            ):
                pred_map_fp32 = model(x_gpu.float())
                pred_sif_fp32 = aggregate_prediction(
                    pred_map_fp32, weight_gpu.float()
                )
                loss_fp32 = F.smooth_l1_loss(
                    pred_sif_fp32,
                    y_gpu.float(),
                    reduction='mean',
                    beta=HUBER_BETA,
                )

            print('First problematic batch:', batch_no)
            print('Input finite:', input_finite)
            print('Weight finite:', weight_finite)
            print('Target finite:', target_finite)
            print('AMP map finite:', amp_map_finite)
            print('AMP prediction finite:', amp_prediction_finite)
            print('AMP loss:', float(loss_amp.detach().cpu()))
            print('FP32 map finite:', tensor_is_finite(pred_map_fp32))
            print(
                'FP32 prediction finite:', tensor_is_finite(pred_sif_fp32)
            )
            print('FP32 loss:', float(loss_fp32.detach().cpu()))
            print('Target range:', float(y.min()), float(y.max()))

            batch_channel_max = pd.DataFrame({
                'channel': channel_names,
                'max_abs_normalized_value': batch_absmax,
            }).sort_values(
                'max_abs_normalized_value', ascending=False
            )
            display(batch_channel_max)
            break

if was_training:
    model.train()

if not bad_batch_found:
    print(
        f'No problematic batches found in {scanned_batches} pre-training batches.'
    )

global_channel_max = pd.DataFrame({
    'channel': channel_names,
    'max_abs_normalized_value': global_channel_absmax,
}).sort_values(
    'max_abs_normalized_value', ascending=False
)

print('Maximum absolute normalized values scanned before stopping:')
display(global_channel_max)

## 11. Training

In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=LR_FACTOR,
    patience=LR_PATIENCE,
    min_lr=MIN_LEARNING_RATE,
)
scaler = torch.cuda.amp.GradScaler(enabled=AMP_ENABLED)

history = []
best_val_rmse = np.inf
best_epoch = 0
best_state = None
epochs_without_improvement = 0

for epoch in range(1, EPOCHS + 1):
    train_batch_sampler.set_epoch(epoch)
    model.train()
    optimizer.zero_grad(set_to_none=True)
    epoch_losses = []

    for step, (x, weight, y, _, _) in enumerate(train_loader, start=1):
        x = x.to(DEVICE, non_blocking=True)
        weight = weight.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        with torch.autocast(
            device_type=DEVICE.type,
            dtype=torch.float16,
            enabled=AMP_ENABLED,
        ):
            pred_map = model(x)
            pred_sif = aggregate_prediction(pred_map, weight)
            loss = F.smooth_l1_loss(
                pred_sif, y, reduction='mean', beta=HUBER_BETA
            )
            accumulated_loss = loss / GRADIENT_ACCUMULATION_STEPS

        scaler.scale(accumulated_loss).backward()
        should_step = (
            step % GRADIENT_ACCUMULATION_STEPS == 0
            or step == len(train_loader)
        )
        if should_step:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        epoch_losses.append(float(loss.detach().cpu()))

    val_predictions_epoch = predict_table(model, val_loader, val_dataset)
    val_metrics = regression_metrics(
        val_predictions_epoch['observed_sif'],
        val_predictions_epoch['predicted_sif_raw'],
    )
    train_loss = float(np.mean(epoch_losses))
    current_lr = float(optimizer.param_groups[0]['lr'])
    scheduler.step(val_metrics['rmse'])

    history.append({
        'epoch': epoch,
        'train_loss': train_loss,
        'val_rmse': val_metrics['rmse'],
        'val_mae': val_metrics['mae'],
        'val_bias': val_metrics['bias'],
        'val_r2': val_metrics['r2'],
        'learning_rate': current_lr,
    })

    print(
        f'Epoch {epoch:03d} | train_huber={train_loss:.5f} | '
        f"val_rmse={val_metrics['rmse']:.5f} | "
        f"val_r2={val_metrics['r2']:.4f} | "
        f"val_mae={val_metrics['mae']:.5f} | "
        f"val_bias={val_metrics['bias']:.5f} | lr={current_lr:.2e}"
    )

    if val_metrics['rmse'] < best_val_rmse:
        best_val_rmse = val_metrics['rmse']
        best_epoch = epoch
        best_state = {
            name: tensor.detach().cpu().clone()
            for name, tensor in model.state_dict().items()
        }
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
        print(f'Early stopping at epoch {epoch}; best epoch was {best_epoch}.')
        break

if best_state is None:
    raise RuntimeError('No best model state was recorded.')

model.load_state_dict(best_state)
history_df = pd.DataFrame(history)
print('best epoch:', best_epoch)
print('best validation RMSE:', best_val_rmse)

## 12. Validation Calibration and Final Test Evaluation

In [ ]:
val_predictions = predict_table(model, val_loader, val_dataset)
test_predictions = predict_table(model, test_loader, test_dataset)

val_metrics_raw = regression_metrics(
    val_predictions['observed_sif'], val_predictions['predicted_sif_raw']
)
test_metrics_raw = regression_metrics(
    test_predictions['observed_sif'], test_predictions['predicted_sif_raw']
)
calibration = fit_linear_calibration(
    val_predictions['observed_sif'].to_numpy(),
    val_predictions['predicted_sif_raw'].to_numpy(),
)

for table in (val_predictions, test_predictions):
    table['predicted_sif_calibrated'] = apply_linear_calibration(
        table['predicted_sif_raw'].to_numpy(), calibration
    )
    table['predicted_sif'] = table['predicted_sif_calibrated']
    table['residual_raw'] = table['predicted_sif_raw'] - table['observed_sif']
    table['residual_calibrated'] = (
        table['predicted_sif_calibrated'] - table['observed_sif']
    )
    table['residual'] = table['residual_calibrated']

val_metrics_calibrated = regression_metrics(
    val_predictions['observed_sif'],
    val_predictions['predicted_sif_calibrated'],
)
test_metrics_calibrated = regression_metrics(
    test_predictions['observed_sif'],
    test_predictions['predicted_sif_calibrated'],
)

print('Validation metrics raw:')
print(val_metrics_raw)
print('\nValidation metrics calibrated:')
print(val_metrics_calibrated)
print('\nTest metrics raw:')
print(test_metrics_raw)
print('\nTest metrics calibrated:')
print(test_metrics_calibrated)
print('\nValidation calibration:')
print(calibration)
display(test_predictions.head())

## 13. Validation Channel Permutation Importance

In [ ]:
PERMUTATION_REPEATS = 3
PERMUTATION_SEED = SEED + 1000


def collect_normalized_channel_maps(
    loader: DataLoader,
    dataset: SentinelAggregateDataset,
    channel_index: int,
) -> np.ndarray:
    channel_maps = None
    for x, _, _, _, sample_indices in loader:
        if channel_maps is None:
            channel_maps = np.empty(
                (len(dataset), x.shape[-2], x.shape[-1]),
                dtype=np.float32,
            )
        channel_maps[sample_indices.numpy()] = x[:, channel_index].numpy()
    if channel_maps is None:
        raise ValueError('Validation loader is empty.')
    return channel_maps


@torch.no_grad()
def predict_with_permuted_channel(
    model: nn.Module,
    loader: DataLoader,
    channel_index: int,
    channel_maps: np.ndarray,
    donor_indices: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    model.eval()
    observed_parts = []
    predicted_parts = []

    for x, weight, y, _, sample_indices in loader:
        sample_indices_np = sample_indices.numpy()
        donor_maps = channel_maps[donor_indices[sample_indices_np]]
        x = x.clone()
        x[:, channel_index] = torch.from_numpy(donor_maps)

        x_gpu = x.to(DEVICE, non_blocking=True)
        weight_gpu = weight.to(DEVICE, non_blocking=True)
        with torch.autocast(
            device_type=DEVICE.type,
            dtype=torch.float16,
            enabled=AMP_ENABLED,
        ):
            pred_map = model(x_gpu)

        predicted_parts.append(
            aggregate_prediction(pred_map, weight_gpu).cpu().numpy()
        )
        observed_parts.append(y.numpy())

    observed = denormalize_sif(np.concatenate(observed_parts))
    predicted = denormalize_sif(np.concatenate(predicted_parts))
    return observed, predicted


baseline_validation_rmse = val_metrics_raw['rmse']
importance_rows = []

for channel_index, channel_name in enumerate(channel_names):
    print(f'Permutation importance: {channel_index + 1} / {len(channel_names)} - {channel_name}')
    channel_maps = collect_normalized_channel_maps(
        val_loader, val_dataset, channel_index
    )

    for repeat in range(PERMUTATION_REPEATS):
        rng = np.random.default_rng(
            PERMUTATION_SEED + channel_index * 100 + repeat
        )
        donor_indices = rng.permutation(len(val_dataset))
        observed_permuted, predicted_permuted = predict_with_permuted_channel(
            model,
            val_loader,
            channel_index,
            channel_maps,
            donor_indices,
        )
        permuted_metrics = regression_metrics(
            observed_permuted, predicted_permuted
        )
        importance_rows.append({
            'channel': channel_name,
            'channel_index': channel_index,
            'repeat': repeat + 1,
            'baseline_validation_rmse': baseline_validation_rmse,
            'permuted_validation_rmse': permuted_metrics['rmse'],
            'delta_rmse': permuted_metrics['rmse'] - baseline_validation_rmse,
        })

    del channel_maps

permutation_importance_repeats = pd.DataFrame(importance_rows)
permutation_importance_summary = (
    permutation_importance_repeats
    .groupby(['channel', 'channel_index'], as_index=False)
    .agg(
        mean_permuted_rmse=('permuted_validation_rmse', 'mean'),
        mean_delta_rmse=('delta_rmse', 'mean'),
        sd_delta_rmse=('delta_rmse', 'std'),
    )
)
permutation_importance_summary['sd_delta_rmse'] = (
    permutation_importance_summary['sd_delta_rmse'].fillna(0.0)
)
permutation_importance_summary['lower_delta_rmse'] = (
    permutation_importance_summary['mean_delta_rmse']
    - permutation_importance_summary['sd_delta_rmse']
)
permutation_importance_summary['upper_delta_rmse'] = (
    permutation_importance_summary['mean_delta_rmse']
    + permutation_importance_summary['sd_delta_rmse']
)
permutation_importance_summary = (
    permutation_importance_summary
    .sort_values('mean_delta_rmse', ascending=False)
    .reset_index(drop=True)
)

display(permutation_importance_summary)

importance_order = permutation_importance_summary['channel'].tolist()
importance_base = alt.Chart(permutation_importance_summary).encode(
    y=alt.Y(
        'channel:N',
        sort=importance_order,
        title='Predictor channel',
    )
)
importance_bars = importance_base.mark_bar().encode(
    x=alt.X(
        'mean_delta_rmse:Q',
        title='Increase in validation RMSE after permutation',
    ),
    color=alt.condition(
        alt.datum.mean_delta_rmse >= 0,
        alt.value('#2c7fb8'),
        alt.value('#bdbdbd'),
    ),
)
importance_errors = importance_base.mark_rule(color='black').encode(
    x=alt.X('lower_delta_rmse:Q'),
    x2=alt.X2('upper_delta_rmse:Q'),
)
importance_points = importance_base.mark_point(
    color='black', filled=True, size=35
).encode(x=alt.X('mean_delta_rmse:Q'))
importance_zero = (
    alt.Chart(pd.DataFrame({'x': [0.0]}))
    .mark_rule(color='black', strokeDash=[4, 4])
    .encode(x='x:Q')
)
importance_chart = (
    importance_bars + importance_errors + importance_points + importance_zero
).properties(
    width=650,
    height=max(360, 24 * len(channel_names)),
    title=(
        'Validation channel permutation importance '
        f'({PERMUTATION_REPEATS} repeats)'
    ),
)
display(importance_chart)

permutation_importance_repeats.to_csv(
    OUTPUT_DIR / 'validation_permutation_importance_repeats.csv',
    index=False,
)
permutation_importance_summary.to_csv(
    OUTPUT_DIR / 'validation_permutation_importance_summary.csv',
    index=False,
)

## 14. Metrics by SIF Bin, Footprint Count, Mode, Month, Year, Tile, State, and Hardiness Zone

In [ ]:
test_predictions['sif_bin'] = pd.cut(
    test_predictions['observed_sif'],
    bins=SIF_BIN_EDGES,
    labels=SIF_BIN_LABELS,
    right=False,
    include_lowest=True,
)


def metrics_by_group(table: pd.DataFrame, group_column: str) -> pd.DataFrame:
    rows = []
    for group_value, group in table.groupby(
        group_column, dropna=False, observed=False
    ):
        raw = regression_metrics(group['observed_sif'], group['predicted_sif_raw'])
        calibrated = regression_metrics(
            group['observed_sif'], group['predicted_sif_calibrated']
        )
        rows.append({
            'group_variable': group_column,
            'group_value': str(group_value),
            'n': raw['n'],
            'observed_min': float(group['observed_sif'].min()),
            'observed_max': float(group['observed_sif'].max()),
            'observed_mean': float(group['observed_sif'].mean()),
            'raw_predicted_mean': float(group['predicted_sif_raw'].mean()),
            'raw_rmse': raw['rmse'],
            'raw_mae': raw['mae'],
            'raw_bias': raw['bias'],
            'raw_r2': raw['r2'],
            'calibrated_predicted_mean': float(
                group['predicted_sif_calibrated'].mean()
            ),
            'calibrated_rmse': calibrated['rmse'],
            'calibrated_mae': calibrated['mae'],
            'calibrated_bias': calibrated['bias'],
            'calibrated_r2': calibrated['r2'],
        })
    return pd.DataFrame(rows)


group_columns = [
    'sif_bin', 'n_footprints', 'measurement_mode', 'month', 'sif_year',
    'mgrs_tile_t', 'states', 'hzs_values'
]
group_metrics = pd.concat(
    [metrics_by_group(test_predictions, column) for column in group_columns],
    ignore_index=True,
)

for column in group_columns:
    print('\n', column)
    display(group_metrics[group_metrics['group_variable'] == column])

## 15. Training and Prediction Diagnostics

In [ ]:
history_long = history_df.melt(
    id_vars=['epoch'],
    value_vars=['train_loss', 'val_rmse'],
    var_name='metric',
    value_name='value',
)
history_chart = (
    alt.Chart(history_long)
    .mark_line(point=True)
    .encode(
        x=alt.X('epoch:Q', title='Epoch'),
        y=alt.Y('value:Q', title='Metric value'),
        color=alt.Color('metric:N', title=None),
    )
    .properties(width=700, height=330, title='Training history')
)
display(history_chart)

observed = test_predictions['observed_sif'].to_numpy(dtype=np.float64)
predicted = test_predictions[
    'predicted_sif_calibrated'
].to_numpy(dtype=np.float64)
plot_valid = np.isfinite(observed) & np.isfinite(predicted)
observed_plot = observed[plot_valid]
predicted_plot = predicted[plot_valid]

axis_min = math.floor(
    min(np.min(observed_plot), np.min(predicted_plot)) * 4
) / 4
axis_max = math.ceil(
    max(np.max(observed_plot), np.max(predicted_plot)) * 4
) / 4
if axis_max <= axis_min:
    axis_max = axis_min + 0.25

TICK_STEP = 0.25
axis_ticks = np.arange(
    axis_min,
    axis_max + TICK_STEP * 0.5,
    TICK_STEP,
).round(10).tolist()
shared_scale = alt.Scale(domain=[axis_min, axis_max], nice=False)
shared_axis = alt.Axis(values=axis_ticks, format='.2f')

# Assign every observation the count of its local 2D histogram bin.
DENSITY_BINS = 60
density_grid, observed_edges, predicted_edges = np.histogram2d(
    observed_plot,
    predicted_plot,
    bins=DENSITY_BINS,
    range=[[axis_min, axis_max], [axis_min, axis_max]],
)
observed_bin = np.clip(
    np.searchsorted(observed_edges, observed_plot, side='right') - 1,
    0,
    DENSITY_BINS - 1,
)
predicted_bin = np.clip(
    np.searchsorted(predicted_edges, predicted_plot, side='right') - 1,
    0,
    DENSITY_BINS - 1,
)
point_density = density_grid[observed_bin, predicted_bin]

density_points = pd.DataFrame({
    'observed_sif': observed_plot,
    'predicted_sif': predicted_plot,
    'density': point_density,
}).sort_values('density', ascending=True)

slope, intercept = np.polyfit(observed_plot, predicted_plot, deg=1)
line_x = np.asarray([axis_min, axis_max], dtype=np.float64)
regression_data = pd.DataFrame({
    'x': line_x,
    'y': intercept + slope * line_x,
})
diagonal_data = pd.DataFrame({
    'x': [axis_min, axis_max],
    'y': [axis_min, axis_max],
})

point_layer = (
    alt.Chart(density_points)
    .mark_circle(size=28, opacity=0.78, clip=True)
    .encode(
        x=alt.X(
            'observed_sif:Q',
            scale=shared_scale,
            axis=shared_axis,
            title='Observed 4 km aggregate SIF',
        ),
        y=alt.Y(
            'predicted_sif:Q',
            scale=shared_scale,
            axis=shared_axis,
            title='Calibrated predicted 4 km aggregate SIF',
        ),
        color=alt.Color(
            'density:Q',
            scale=alt.Scale(type='log', scheme='turbo', domainMin=1),
            title='Local density',
        ),
        tooltip=[
            alt.Tooltip('observed_sif:Q', format='.4f'),
            alt.Tooltip('predicted_sif:Q', format='.4f'),
            alt.Tooltip('density:Q', format='.0f'),
        ],
    )
)

diagonal = (
    alt.Chart(diagonal_data)
    .mark_line(color='black', strokeWidth=1.5, strokeDash=[6, 4])
    .encode(
        x=alt.X('x:Q', scale=shared_scale),
        y=alt.Y('y:Q', scale=shared_scale),
    )
)
regression_line = (
    alt.Chart(regression_data)
    .mark_line(color='#e31a1c', strokeWidth=2.0)
    .encode(
        x=alt.X('x:Q', scale=shared_scale),
        y=alt.Y('y:Q', scale=shared_scale),
    )
)

axis_span = axis_max - axis_min
annotation_data = pd.DataFrame({
    'x': [
        axis_min + 0.04 * axis_span,
        axis_min + 0.04 * axis_span,
    ],
    'y': [
        axis_max - 0.04 * axis_span,
        axis_max - 0.09 * axis_span,
    ],
    'label': [
        f'regression: y={slope:.3f}x{intercept:+.3f}',
        (
            f"RMSE={test_metrics_calibrated['rmse']:.4f}, "
            f"R2={test_metrics_calibrated['r2']:.3f}, "
            f"N={len(density_points):,}"
        ),
    ],
})
annotation = (
    alt.Chart(annotation_data)
    .mark_text(
        align='left',
        baseline='top',
        color='black',
        fontSize=13,
    )
    .encode(
        x=alt.X('x:Q', scale=shared_scale),
        y=alt.Y('y:Q', scale=shared_scale),
        text='label:N',
    )
)

density_title = (
    f"Test RMSE={test_metrics_calibrated['rmse']:.4f}, "
    f"R2={test_metrics_calibrated['r2']:.3f}"
)
density_chart = (
    (point_layer + diagonal + regression_line + annotation)
    .properties(width=620, height=620, title=density_title)
    .configure_view(strokeWidth=1)
)
display(density_chart)

residual_chart = (
    alt.Chart(test_predictions)
    .mark_bar()
    .encode(
        x=alt.X(
            'residual_calibrated:Q',
            bin=alt.Bin(maxbins=80),
            title='Calibrated predicted - observed SIF',
        ),
        y=alt.Y('count():Q', title='Count'),
    )
    .properties(width=700, height=320, title='Test residual distribution')
)
display(residual_chart)

## 16. Example Predicted SIF Map and Aggregate Supervision

In [ ]:
example_index = 0
raw_example_dataset = SentinelAggregateDataset(
    test_table.iloc[[example_index]].reset_index(drop=True), cache_size=1
)
normalized_example_dataset = SentinelAggregateDataset(
    test_table.iloc[[example_index]].reset_index(drop=True), **dataset_kwargs
)

raw_x, raw_weight, raw_y, raw_n, _ = raw_example_dataset[0]
norm_x, norm_weight, norm_y, norm_n, _ = normalized_example_dataset[0]

model.eval()
with torch.no_grad():
    with torch.autocast(
        device_type=DEVICE.type,
        dtype=torch.float16,
        enabled=AMP_ENABLED,
    ):
        example_pred_norm = model(norm_x[None].to(DEVICE))[0, 0]
    example_pred_map_raw = denormalize_sif(
        example_pred_norm.float().cpu().numpy()
    )
    example_pred_map = apply_linear_calibration(
        example_pred_map_raw, calibration
    )

ndvi_index = channel_names.index('ndvi')
figure, axes = plt.subplots(1, 3, figsize=(18, 5.5), constrained_layout=True)

image0 = axes[0].imshow(example_pred_map, cmap='viridis')
axes[0].set_title('Calibrated predicted 20 m SIF map')
figure.colorbar(image0, ax=axes[0], fraction=0.046)

image1 = axes[1].imshow(raw_x[ndvi_index], cmap='RdYlGn', vmin=-1, vmax=1)
axes[1].set_title('Raw NDVI channel')
figure.colorbar(image1, ax=axes[1], fraction=0.046)

image2 = axes[2].imshow(raw_weight, cmap='magma')
axes[2].set_title(f'Aggregate supervision weights (n={int(raw_n)})')
figure.colorbar(image2, ax=axes[2], fraction=0.046)

for axis in axes:
    axis.set_xticks([])
    axis.set_yticks([])
plt.show()

example_predicted = float((example_pred_map * raw_weight.numpy()).sum())
example_observed = float(raw_y)
display(pd.DataFrame([{
    'aggregation_id': test_table.iloc[example_index]['aggregation_id'],
    'n_footprints': int(raw_n),
    'observed_aggregate_sif': example_observed,
    'predicted_aggregate_sif_calibrated': example_predicted,
    'residual': example_predicted - example_observed,
}]))

## 17. Save Model, Predictions, Splits, and Metrics

In [ ]:
history_path = OUTPUT_DIR / 'training_history.csv'
val_predictions_path = OUTPUT_DIR / 'validation_predictions.csv'
test_predictions_path = OUTPUT_DIR / 'test_predictions.csv'
group_metrics_path = OUTPUT_DIR / 'test_group_metrics.csv'
split_path = OUTPUT_DIR / 'aggregate_chip_splits.csv'
component_path = OUTPUT_DIR / 'split_components.csv'
checkpoint_path = OUTPUT_DIR / 'sentinel2_spatial_aggregate_4km_unet.pt'
metrics_path = OUTPUT_DIR / 'metrics.json'

history_df.to_csv(history_path, index=False)
val_predictions.to_csv(val_predictions_path, index=False)
test_predictions.to_csv(test_predictions_path, index=False)
group_metrics.to_csv(group_metrics_path, index=False)

chip_splits = pd.concat([
    train_table,
    val_table,
    test_table,
], ignore_index=True)[
    [
        'aggregation_id', 'Delta_Date', 'product_path', 'split_component',
        'split', 'mgrs_tile_t', 'measurement_mode', 'n_footprints'
    ]
]
chip_splits.to_csv(split_path, index=False)
split_components.to_csv(component_path, index=False)

checkpoint = {
    'model_state_dict': best_state,
    'model_class': 'ThreeLevelUNet',
    'channel_names': channel_names,
    'target_column': TARGET_COL,
    'base_channels': BASE_CHANNELS,
    'channel_mean': channel_mean,
    'channel_std': channel_std,
    'target_mean': target_mean,
    'target_std': target_std,
    'normalize_target': NORMALIZE_TARGET,
    'calibration': calibration,
    'best_epoch': best_epoch,
    'best_validation_rmse': best_val_rmse,
    'config': {
        'batch_size': BATCH_SIZE,
        'gradient_accumulation_steps': GRADIENT_ACCUMULATION_STEPS,
        'epochs': EPOCHS,
        'learning_rate': LEARNING_RATE,
        'weight_decay': WEIGHT_DECAY,
        'huber_beta': HUBER_BETA,
        'seed': SEED,
        'split_grouping': ['Delta_Date', 'product_path'],
        'split_balance_marginals': [
            'month', 'measurement_mode', 'mgrs_tile_t'
        ],
        'split_allocation': 'chip_count_aware_component_search',
        'max_samples_for_stats': MAX_SAMPLES_FOR_STATS,
        'amp_enabled': AMP_ENABLED,
        'equal_cell_loss_weight': True,
    },
}
torch.save(checkpoint, checkpoint_path)

metrics_payload = {
    'best_epoch': best_epoch,
    'best_validation_rmse_during_training': best_val_rmse,
    'validation_raw': val_metrics_raw,
    'validation_calibrated': val_metrics_calibrated,
    'test_raw': test_metrics_raw,
    'test_calibrated': test_metrics_calibrated,
    'calibration': calibration,
    'n_train_chips': len(train_table),
    'n_validation_chips': len(val_table),
    'n_test_chips': len(test_table),
    'n_train_dates': int(train_table['Delta_Date'].nunique()),
    'n_validation_dates': int(val_table['Delta_Date'].nunique()),
    'n_test_dates': int(test_table['Delta_Date'].nunique()),
    'n_train_products': int(train_table['product_path'].nunique()),
    'n_validation_products': int(val_table['product_path'].nunique()),
    'n_test_products': int(test_table['product_path'].nunique()),
    'stats_cache_path': str(stats_cache_path),
}
with metrics_path.open('w', encoding='utf-8') as file:
    json.dump(metrics_payload, file, indent=2)

print('Saved:')
for path in [
    history_path, val_predictions_path, test_predictions_path,
    group_metrics_path, split_path, component_path,
    checkpoint_path, metrics_path,
]:
    print(' -', path)